# Fine-tuning BLIP — descrição de condição do livro

Antes de rodar:

1. **Add Input** (barra lateral) → anexar o Kaggle Dataset das fotos:
   `pedrocanoas/fotos-livros`. O código vem direto do GitHub (célula 1),
   não precisa mais de um segundo Dataset.
2. **Settings** → Accelerator: **GPU (T4 x2 ou P100)** · Internet: **On**
   (necessário tanto pro `git clone` quanto pra baixar os pesos do BLIP do
   Hugging Face na primeira execução)

> **Nota sobre o layout do Kaggle:** o Kaggle monta os datasets em
> `/kaggle/input/datasets/<usuário>/<slug>/...` (não mais direto em
> `/kaggle/input/<slug>/...`). Confirme rodando `!find /kaggle/input -maxdepth 4`
> se os caminhos abaixo não baterem com a estrutura do seu notebook.

In [ ]:
# Célula 1 — clona o código direto do GitHub (sempre pega a versão mais recente do main)
!git clone --depth 1 https://github.com/Pedrocanoas/tcc.git /kaggle/working/repo
%cd /kaggle/working/repo/model

In [ ]:
# Célula 2 — instala as deps que faltam (não reinstala torch: o Kaggle já vem com build CUDA pronta)
!pip install -q -r requirements-kaggle.txt

In [ ]:
# Célula 3 — aponta os diretórios de dados/checkpoints para /kaggle/working (única área gravável)
import os

os.environ["TCC_DATASET_RAW_DIR"] = "/kaggle/input/datasets/pedrocanoas/fotos-livros"
os.environ["TCC_DATASET_PROCESSED_DIR"] = "/kaggle/working/processed"
os.environ["TCC_CHECKPOINTS_DIR"] = "/kaggle/working/checkpoints"

In [ ]:
# Célula 4 — gera os manifestos (train.jsonl / val.jsonl) e roda o fine-tuning
# Com ~2975 imagens numa T4 isso roda em minutos; se faltar VRAM, reduza --batch-size (2 ou 1)
# --max-length 128: com 64 (default anterior), 77% das legendas eram cortadas no treino
# e o modelo nunca aprendia a parar de gerar texto. 128 cobre ~91% das legendas inteiras.
# --exclude-colecoes: tira do TREINO os ~5,8% de livros que são coleções/multi-volume
# (ver README) — mantida porque o checkpoint de produção atual foi treinado com ela.
#
# Experimento: encoder parcialmente destravado. Destravar o encoder inteiro com lr 5e-5
# deu mode collapse, e 576 com o encoder todo congelado piorou (0.31 vs 0.34) — ver
# model/README.md. Aqui só as 2 últimas camadas do encoder (+ tabela de posições e
# post_layernorm) treinam, com lr 5x menor que o decoder. Dois treinos em sequência:
#   A) 576 + 2 camadas destravadas
#   B) 384 + 2 camadas destravadas (controle: separa o efeito da resolução do efeito
#      de destravar)
COMMON = "--raw-dir /kaggle/input/datasets/pedrocanoas/fotos-livros --epochs 20 --batch-size 8 --max-length 128 --exclude-colecoes --unfreeze-vision-layers 2 --vision-lr 1e-5"
!python kaggle_train.py {COMMON} --image-size 576 --checkpoints-dir /kaggle/working/ckpt_576_unf2
!python kaggle_train.py {COMMON} --checkpoints-dir /kaggle/working/ckpt_384_unf2

In [ ]:
# Célula 5 (opcional) — sanity check: gera uma legenda de exemplo com o checkpoint treinado
import glob

sample_photos = sorted(glob.glob("/kaggle/input/datasets/pedrocanoas/fotos-livros/livro001/foto*.jpg"))[:1]
if sample_photos:
    !python -m src.infer {sample_photos[0]}
else:
    print("ajuste o caminho de exemplo acima para uma foto existente no seu dataset")

In [ ]:
# Célula 6 — remove os checkpoints "last" (só o "best" interessa) pra encurtar o download
# do Output; as pastas best/ já aparecem no Output sem precisar de zip
!rm -rf /kaggle/working/ckpt_*/last
!du -sh /kaggle/working/ckpt_*/best

Depois de rodar tudo, clique em **Save Version** (Save & Run All) para commitar o
notebook — as pastas `ckpt_*/best/` aparecem na aba **Output** para baixar direto
pelo navegador, ou via Kaggle API:

```sh
kaggle kernels output <seu-usuario>/<slug-do-notebook> -p ./kaggle-output
```

Copie a pasta `best/` escolhida para `model/checkpoints/best/` (caminho que
`infer.py` usa por padrão localmente).